# Experimento 04 — Compliance Check de Proveedores
**PROJENER.AI SL · UAX · Máster Big Data · 2026**  
Autora: Edurne Martínez de Contrasta

Auditoría periódica de cumplimiento normativo de proveedores activos en ACME Industrial S.A.  
Mismos modelos y métricas que Experimentos 01, 02 y 03.

**Dataset:** 20 casos — Simple (C01-C08), Medio (C09-C14), Complejo (C15-C20)  
**6 casos requieren HiL** — todos Complejo: sanción REACH, brecha GDPR, accidente laboral grave, alerta OFAC, conflicto de interés accionista, fraude en facturación  
**Agentes:** Auditor, Legal, Compliance, Finanzas  
**Modelos:** M1 (RPA baseline), M2 (Single Agent 8b), M4 (Multi-Agent mixto)

In [6]:
import sys, re, time, json, os
os.environ["GROQ_API_KEY"] = "API_KEY"
os.environ["OTEL_SDK_DISABLED"] = "true"

print(f"Python: {sys.version[:6]}")
print(f"Dataset: {os.path.exists('casos_compliance.json')}")

Python: 3.12.1
Dataset: True


## APIs Mock — Compliance Check

In [2]:
def verificar_certificaciones(proveedor_id, sector, certificaciones_vigentes, iso_9001):
    """API mock — estado de certificaciones del proveedor"""
    requiere_iso = sector in ["tecnologia", "manufactura", "equipamiento", "quimica"]
    return {
        "proveedor_id": proveedor_id,
        "certificaciones_vigentes": certificaciones_vigentes,
        "iso_9001": iso_9001,
        "requiere_iso": requiere_iso,
        "alerta_certificacion": not certificaciones_vigentes or (requiere_iso and not iso_9001),
        "nivel_riesgo": "alto" if not certificaciones_vigentes else "medio" if requiere_iso and not iso_9001 else "bajo"
    }

def verificar_compliance_normativo(caso):
    """API mock — verificación de cumplimiento normativo"""
    sancion      = caso.get("sancion_regulatoria", False)
    brecha       = caso.get("brecha_datos", False)
    accidente    = caso.get("accidente_laboral_grave", False)
    sanciones    = caso.get("alerta_sanciones_internacionales", False)
    conflicto    = caso.get("conflicto_interes", False)
    fraude       = caso.get("facturacion_fraudulenta", False)
    gdpr         = caso.get("gdpr_cumple", True)
    prl          = caso.get("prl_cumple", True)

    nivel = "critico" if (sancion or brecha or accidente or sanciones or fraude) else             "alto"    if (conflicto or not gdpr) else             "medio"   if not prl else "bajo"

    return {
        "sancion_regulatoria": sancion,
        "brecha_datos_gdpr": brecha,
        "accidente_laboral_grave": accidente,
        "alerta_sanciones_internacionales": sanciones,
        "conflicto_interes": conflicto,
        "facturacion_fraudulenta": fraude,
        "gdpr_cumple": gdpr,
        "prl_cumple": prl,
        "nivel_riesgo_compliance": nivel,
        "requiere_hil": nivel in ["critico", "alto"],
        "accion_recomendada": "bloquear_escalar" if nivel == "critico" else
                              "revision_urgente" if nivel == "alto" else
                              "renovacion_condicional" if nivel == "medio" else "renovar"
    }

def verificar_historial_financiero(proveedor_id, volumen_anual_eur, pagos_al_dia, incidencias):
    """API mock — historial financiero y de incidencias"""
    riesgo = "alto"  if not pagos_al_dia or incidencias >= 4 else              "medio" if incidencias >= 2 else "bajo"
    return {
        "proveedor_id": proveedor_id,
        "volumen_anual_eur": volumen_anual_eur,
        "pagos_al_dia": pagos_al_dia,
        "incidencias_ultimo_año": incidencias,
        "riesgo_financiero": riesgo,
        "renovacion_recomendada": riesgo != "alto"
    }

def verificar_legal_contrato(proveedor_id, años_proveedor, cambio_societario=False):
    """API mock — estado legal y contractual"""
    return {
        "proveedor_id": proveedor_id,
        "años_relacion": años_proveedor,
        "cambio_societario": cambio_societario,
        "contrato_vigente": True,
        "requiere_actualizacion": cambio_societario or años_proveedor > 5,
        "riesgo_legal": "medio" if cambio_societario else "bajo"
    }

def registrar_auditoria(caso):
    """API mock — registro de auditoría"""
    return {
        "auditoria_id": f"AUD-2026-{caso['id']}",
        "proveedor_id": caso.get("proveedor_id", ""),
        "nombre": caso.get("nombre", ""),
        "estado": "en_revision",
        "fecha": "2026-05-01"
    }

print("✅ APIs mock cargadas")

✅ APIs mock cargadas


## Dataset — 20 Casos de Compliance Check

In [3]:
with open("casos_compliance.json", encoding="utf-8") as f:
    data = json.load(f)
casos_comp = data["casos"]

print(f"✅ {len(casos_comp)} casos cargados")
print(f"   Simple:   {sum(1 for c in casos_comp if c['nivel']=='Simple')}")
print(f"   Medio:    {sum(1 for c in casos_comp if c['nivel']=='Medio')}")
print(f"   Complejo: {sum(1 for c in casos_comp if c['nivel']=='Complejo')}")
print(f"   Requieren HiL: {sum(1 for c in casos_comp if c['ground_truth']['requiere_hil'])}")

✅ 20 casos cargados
   Simple:   8
   Medio:    6
   Complejo: 6
   Requieren HiL: 6


In [4]:
def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados if not r["escalo_hil"] and r["decision"] != "error")
    hil_req   = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det   = [r for r in hil_req    if r["escalo_hil"]]
    errores   = [r for r in resultados if r["decision"] != r["ground_truth"].get("resultado","")
                 and not r["escalo_hil"] and r["decision"] != "error"]
    ARR = round(resueltos / total * 100, 1)
    HDA = round(len(hil_det) / len(hil_req) * 100, 1) if hil_req else 0
    DER = round(len(errores) / total * 100, 1)
    PT  = round(sum(r["tiempo"] for r in resultados) / total, 2)
    return ARR, HDA, DER, PT

os.makedirs("resultados_compliance", exist_ok=True)
print("✅ Función métricas lista")

✅ Función métricas lista


## M1 — Baseline RPA

In [5]:
def procesar_compliance_m1(caso):
    """Baseline RPA — reglas if-then para compliance check"""
    sancion   = caso.get("sancion_regulatoria", False)
    brecha    = caso.get("brecha_datos", False)
    accidente = caso.get("accidente_laboral_grave", False)
    sanciones = caso.get("alerta_sanciones_internacionales", False)
    fraude    = caso.get("facturacion_fraudulenta", False)
    conflicto = caso.get("conflicto_interes", False)
    gdpr      = caso.get("gdpr_cumple", True)
    prl       = caso.get("prl_cumple", True)
    certs     = caso.get("certificaciones_vigentes", True)

    if sancion or brecha or accidente or sanciones or fraude:
        decision = "bloqueo_critico"
    elif conflicto or not gdpr:
        decision = "renovacion_bloqueada"
    elif not prl or not certs:
        decision = "renovacion_condicional"
    else:
        decision = "renovacion_aprobada"

    import time
    return {
        "caso_id": caso["id"], "nivel": caso["nivel"],
        "decision": decision, "escalo_hil": False,
        "tiempo": 0.0, "ground_truth": caso["ground_truth"]
    }

print("Ejecutando M1 — Baseline RPA (20 casos)...\n")
resultados_comp_m1 = []
for caso in casos_comp:
    r = procesar_compliance_m1(caso)
    resultados_comp_m1.append(r)
    gt = r["ground_truth"]["resultado"]
    ok = "✅" if r["decision"] == gt else "❌"
    print(f"  {caso['id']} [{caso['nivel']}]: {r['decision']} | GT: {gt} {ok}")

ARR,HDA,DER,PT = calcular_metricas(resultados_comp_m1, 20)
print(f"\n{'='*40}")
print(f"M1 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s")
print(f"{'='*40}")
with open("resultados_compliance/resultados_comp_m1.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M1_RPA_compliance","metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},
               "resultados_por_caso":resultados_comp_m1}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_comp_m1.json")

Ejecutando M1 — Baseline RPA (20 casos)...

  C01 [Simple]: renovacion_aprobada | GT: renovacion_aprobada ✅
  C02 [Simple]: renovacion_aprobada | GT: renovacion_aprobada ✅
  C03 [Simple]: renovacion_aprobada | GT: renovacion_aprobada_sin_iso ❌
  C04 [Simple]: renovacion_aprobada | GT: renovacion_aprobada_con_nota ❌
  C05 [Simple]: renovacion_aprobada | GT: renovacion_aprobada ✅
  C06 [Simple]: renovacion_aprobada | GT: renovacion_aprobada ✅
  C07 [Simple]: renovacion_aprobada | GT: renovacion_aprobada ✅
  C08 [Simple]: renovacion_aprobada | GT: renovacion_aprobada ✅
  C09 [Medio]: renovacion_condicional | GT: renovacion_condicional_pendiente_iso27001 ❌
  C10 [Medio]: renovacion_condicional | GT: renovacion_condicional_actualizar_prl ❌
  C11 [Medio]: renovacion_bloqueada | GT: renovacion_bloqueada_revision_gdpr ❌
  C12 [Medio]: renovacion_aprobada | GT: renovacion_condicional_regularizar_deuda ❌
  C13 [Medio]: renovacion_aprobada | GT: renovacion_condicional_verificar_administrador ❌
  

M1 resuelve todo mecánicamente (ARR=100%) pero no detecta ningún caso HiL (HDA=0%) y comete errores en casos complejos (DER=70%). El RPA no distingue matices: aprueba todo lo que no tiene alerta binaria explícita.

## M2 — Single Agent (llama-3.1-8b-instant)

In [7]:
import json, sys, re, time, os
from crewai import Agent, Task, Crew, LLM

# Cargar dataset
with open("casos_compliance.json", encoding="utf-8") as f:
    data = json.load(f)
casos_comp = data["casos"]

# APIs mock
def verificar_certificaciones(proveedor_id, sector, certificaciones_vigentes, iso_9001):
    requiere_iso = sector in ["tecnologia", "manufactura", "equipamiento", "quimica"]
    return {"proveedor_id": proveedor_id, "certificaciones_vigentes": certificaciones_vigentes,
            "iso_9001": iso_9001, "requiere_iso": requiere_iso,
            "alerta_certificacion": not certificaciones_vigentes or (requiere_iso and not iso_9001),
            "nivel_riesgo": "alto" if not certificaciones_vigentes else "medio" if requiere_iso and not iso_9001 else "bajo"}

def verificar_compliance_normativo(caso):
    sancion=caso.get("sancion_regulatoria",False); brecha=caso.get("brecha_datos",False)
    accidente=caso.get("accidente_laboral_grave",False); sanciones=caso.get("alerta_sanciones_internacionales",False)
    conflicto=caso.get("conflicto_interes",False); fraude=caso.get("facturacion_fraudulenta",False)
    gdpr=caso.get("gdpr_cumple",True); prl=caso.get("prl_cumple",True)
    nivel = "critico" if (sancion or brecha or accidente or sanciones or fraude) else \
            "alto" if (conflicto or not gdpr) else "medio" if not prl else "bajo"
    return {"sancion_regulatoria":sancion,"brecha_datos_gdpr":brecha,"accidente_laboral_grave":accidente,
            "alerta_sanciones_internacionales":sanciones,"conflicto_interes":conflicto,
            "facturacion_fraudulenta":fraude,"gdpr_cumple":gdpr,"prl_cumple":prl,
            "nivel_riesgo_compliance":nivel,"requiere_hil":nivel in ["critico","alto"],
            "accion_recomendada":"bloquear_escalar" if nivel=="critico" else "revision_urgente" if nivel=="alto" else "renovacion_condicional" if nivel=="medio" else "renovar"}

def verificar_historial_financiero(proveedor_id, volumen_anual_eur, pagos_al_dia, incidencias):
    riesgo = "alto" if not pagos_al_dia or incidencias>=4 else "medio" if incidencias>=2 else "bajo"
    return {"proveedor_id":proveedor_id,"volumen_anual_eur":volumen_anual_eur,
            "pagos_al_dia":pagos_al_dia,"incidencias_ultimo_año":incidencias,
            "riesgo_financiero":riesgo,"renovacion_recomendada":riesgo!="alto"}

def verificar_legal_contrato(proveedor_id, años_proveedor, cambio_societario=False):
    return {"proveedor_id":proveedor_id,"años_relacion":años_proveedor,
            "cambio_societario":cambio_societario,"contrato_vigente":True,
            "requiere_actualizacion":cambio_societario or años_proveedor>5,
            "riesgo_legal":"medio" if cambio_societario else "bajo"}

def registrar_auditoria(caso):
    return {"auditoria_id":f"AUD-2026-{caso['id']}","proveedor_id":caso.get("proveedor_id",""),
            "nombre":caso.get("nombre",""),"estado":"en_revision","fecha":"2026-05-01"}

def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados if not r["escalo_hil"] and r["decision"]!="error")
    hil_req = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det = [r for r in hil_req if r["escalo_hil"]]
    errores = [r for r in resultados if r["decision"]!=r["ground_truth"].get("resultado","") and not r["escalo_hil"] and r["decision"]!="error"]
    ARR = round(resueltos/total*100,1); HDA = round(len(hil_det)/len(hil_req)*100,1) if hil_req else 0
    DER = round(len(errores)/total*100,1); PT = round(sum(r["tiempo"] for r in resultados)/total,2)
    return ARR, HDA, DER, PT

def procesar_compliance_m2(caso):
    info_certs = verificar_certificaciones(caso["proveedor_id"],caso.get("sector",""),caso.get("certificaciones_vigentes",True),caso.get("iso_9001",True))
    info_comp = verificar_compliance_normativo(caso)
    info_fin = verificar_historial_financiero(caso["proveedor_id"],caso.get("volumen_anual_eur",0),caso.get("pagos_al_dia",True),caso.get("incidencias_ultimo_año",0))
    info_leg = verificar_legal_contrato(caso["proveedor_id"],caso.get("años_como_proveedor",0),caso.get("cambio_societario",False))
    registro = registrar_auditoria(caso)
    llm_8b = LLM(model="groq/llama-3.1-8b-instant", temperature=0.0)
    agente = Agent(
        role="Auditor de Compliance",
        goal="Auditar el cumplimiento normativo de proveedores activos de ACME Industrial S.A.",
        backstory=f"""Eres el auditor de compliance de ACME Industrial S.A.
Proveedor: {caso['descripcion']}
Certificaciones: {json.dumps(info_certs, ensure_ascii=False)}
Compliance: {json.dumps(info_comp, ensure_ascii=False)}
Financiero: {json.dumps(info_fin, ensure_ascii=False)}
Legal: {json.dumps(info_leg, ensure_ascii=False)}
Responde SOLO con JSON: {{"decision": "renovacion_aprobada"|"renovacion_condicional"|"renovacion_bloqueada"|"escalacion_hil", "razon": "texto breve", "escala_hil": true|false}}
REGLA: escala_hil=true SOLO si hay sanción regulatoria, brecha de datos, accidente laboral grave, alerta OFAC, conflicto de interés o fraude.""",
        llm=llm_8b, verbose=False)
    tarea = Task(description=f"Audita: {caso['nombre']} — {caso['descripcion'][:200]}", agent=agente, expected_output='JSON con decision, razon, escala_hil')
    crew = Crew(agents=[agente], tasks=[tarea], verbose=False)
    try:
        t_ini=time.time(); resultado=crew.kickoff(); t_fin=time.time()
        texto=str(resultado); decision="error"; escala=False
        try:
            match=re.search(r'\{[^{}]*"decision"[^{}]*\}',texto,re.DOTALL)
            if match:
                datos=json.loads(match.group()); decision=datos.get("decision","error"); escala=datos.get("escala_hil",False)
        except: pass
        if decision=="error":
            tl=texto.lower()
            if "escala" in tl or "hil" in tl: decision="escalacion_hil"; escala=True
            elif "bloquea" in tl: decision="renovacion_bloqueada"
            elif "condicion" in tl: decision="renovacion_condicional"
            else: decision="renovacion_aprobada"
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":decision,"escalo_hil":escala,"tiempo":round(t_fin-t_ini,2),"ground_truth":caso["ground_truth"]}
    except Exception as e:
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":"error","escalo_hil":False,"tiempo":0,"ground_truth":caso["ground_truth"],"error":str(e)[:200]}

os.makedirs("resultados_compliance", exist_ok=True)
print("Ejecutando M2 — Single Agent (20 casos)...\n")
resultados_comp_m2 = []
for i, caso in enumerate(casos_comp):
    print(f"  {caso['id']}...", end=" ", flush=True)
    for intento in range(3):
        r = procesar_compliance_m2(caso)
        if r["decision"] != "error": break
        espera = 60*(intento+1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_comp_m2.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(60)

ARR,HDA,DER,PT = calcular_metricas(resultados_comp_m2, 20)
print(f"\n{'='*40}\nM2 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s\n{'='*40}")
with open("resultados_compliance/resultados_comp_m2.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M2_single_compliance","metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},"resultados_por_caso":resultados_comp_m2}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_comp_m2.json")

Ejecutando M2 — Single Agent (20 casos)...

  C01... renovacion_aprobada (auto)
  C02... renovacion_aprobada (auto)
  C03... renovacion_aprobada (auto)
  C04... renovacion_aprobada (auto)
  C05... 
    Reintento 1 — esperando 60s... renovacion_aprobada (auto)
  C06... renovacion_aprobada (auto)
  C07... renovacion_aprobada (auto)
  C08... renovacion_aprobada (auto)
  C09... renovacion_condicional (auto)
  C10... renovacion_condicional (auto)
  C11... renovacion_condicional (HiL)
  C12... renovacion_condicional (auto)
  C13... renovacion_condicional (auto)
  C14... renovacion_condicional (auto)
  C15... renovacion_condicional (HiL)
  C16... renovacion_bloqueada (HiL)
  C17... renovacion_condicional (auto)
  C18... renovacion_bloqueada (HiL)
  C19... renovacion_bloqueada (HiL)
  C20... renovacion_bloqueada (HiL)

M2 — ARR=70.0% | HDA=83.3% | DER=40.0% | PT=0.6s
✅ Guardado resultados_comp_m2.json


M2 detecta 5 de 6 casos HiL (HDA=83.3%). El caso no detectado (C17: accidente laboral grave) requiere razonamiento transversal — señal implícita que el agente único no siempre integra. Las señales explícitas (sanciones OFAC, brecha GDPR, fraude) se detectan correctamente.

## M4 — Multi-Agent Mixto (8b×3 + 70b×1)

In [8]:
def procesar_compliance_m4(caso):
    info_certs     = verificar_certificaciones(caso["proveedor_id"], caso.get("sector",""),
                                               caso.get("certificaciones_vigentes",True),
                                               caso.get("iso_9001",True))
    info_comp      = verificar_compliance_normativo(caso)
    info_financiero= verificar_historial_financiero(caso["proveedor_id"],
                                                    caso.get("volumen_anual_eur",0),
                                                    caso.get("pagos_al_dia",True),
                                                    caso.get("incidencias_ultimo_año",0))
    info_legal     = verificar_legal_contrato(caso["proveedor_id"],
                                              caso.get("años_como_proveedor",0),
                                              caso.get("cambio_societario",False))
    registro       = registrar_auditoria(caso)

    llm_8b  = LLM(model="groq/llama-3.1-8b-instant",  temperature=0.0)
    llm_70b = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.0)

    auditor = Agent(role="Auditor Interno", goal="Revisar el historial y certificaciones del proveedor.",
                    backstory=f"Proveedor: {caso['descripcion']}\nCertificaciones: {json.dumps(info_certs, ensure_ascii=False)}",
                    llm=llm_8b, verbose=False)
    legal   = Agent(role="Asesor Legal", goal="Revisar el estado legal y contractual.",
                    backstory=f"Legal: {json.dumps(info_legal, ensure_ascii=False)}",
                    llm=llm_8b, verbose=False)
    finanzas= Agent(role="Analista Financiero", goal="Evaluar el historial financiero y de incidencias.",
                    backstory=f"Financiero: {json.dumps(info_financiero, ensure_ascii=False)}",
                    llm=llm_8b, verbose=False)
    compliance = Agent(
        role="Director de Compliance (KYC/AML/Normativa)",
        goal="Tomar la decisión final de renovación basándose en todos los factores de riesgo.",
        backstory=f"""Eres el decisor final de compliance. Tienes toda la información:
Proveedor: {caso['descripcion']}
Compliance: {json.dumps(info_comp, ensure_ascii=False)}
Financiero: {json.dumps(info_financiero, ensure_ascii=False)}
Legal: {json.dumps(info_legal, ensure_ascii=False)}
Certificaciones: {json.dumps(info_certs, ensure_ascii=False)}

Responde SOLO con JSON:
{{"decision": "renovacion_aprobada"|"renovacion_condicional"|"renovacion_bloqueada"|"escalacion_hil",
  "razon": "texto breve",
  "escala_hil": true|false}}

REGLA CRÍTICA: escala_hil=true OBLIGATORIO si:
- sancion_regulatoria=true, brecha_datos_gdpr=true, accidente_laboral_grave=true
- alerta_sanciones_internacionales=true, conflicto_interes=true, facturacion_fraudulenta=true""",
        llm=llm_70b, verbose=False)

    t1 = Task(description=f"Revisa certificaciones de {caso['nombre']}", agent=auditor, expected_output="Evaluación certificaciones")
    t2 = Task(description="Revisa estado legal y contractual", agent=legal, expected_output="Evaluación legal")
    t3 = Task(description="Evalúa historial financiero e incidencias", agent=finanzas, expected_output="Evaluación financiera")
    t4 = Task(description="Decide en JSON la renovación del proveedor", agent=compliance, expected_output="JSON con decision, razon, escala_hil")

    crew = Crew(agents=[auditor, legal, finanzas, compliance], tasks=[t1,t2,t3,t4], verbose=False)
    try:
        t_ini = time.time()
        resultado = crew.kickoff()
        t_fin = time.time()
        texto = str(resultado)
        decision = "error"; escala = False
        try:
            match = re.search(r'\{[^{}]*"decision"[^{}]*\}', texto, re.DOTALL)
            if match:
                datos = json.loads(match.group())
                decision = datos.get("decision","error")
                escala = datos.get("escala_hil",False)
        except: pass
        if decision == "error":
            tl = texto.lower()
            if "escala" in tl or "hil" in tl: decision="escalacion_hil"; escala=True
            elif "bloquea" in tl: decision="renovacion_bloqueada"
            elif "condicion" in tl: decision="renovacion_condicional"
            else: decision="renovacion_aprobada"
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":decision,
                "escalo_hil":escala,"tiempo":round(t_fin-t_ini,2),"ground_truth":caso["ground_truth"]}
    except Exception as e:
        return {"caso_id":caso["id"],"nivel":caso["nivel"],"decision":"error",
                "escalo_hil":False,"tiempo":0,"ground_truth":caso["ground_truth"],"error":str(e)[:200]}

print("Ejecutando M4 — Multi-Agent mixto (20 casos)...\n")
resultados_comp_m4 = []
for i, caso in enumerate(casos_comp):
    print(f"  {caso['id']}...", end=" ", flush=True)
    for intento in range(3):
        r = procesar_compliance_m4(caso)
        if r["decision"] != "error": break
        espera = 70*(intento+1)
        print(f"\n    Reintento {intento+1} — esperando {espera}s...", end=" ")
        time.sleep(espera)
    resultados_comp_m4.append(r)
    print(f"{r['decision']} ({'HiL' if r['escalo_hil'] else 'auto'})")
    time.sleep(70)

ARR,HDA,DER,PT = calcular_metricas(resultados_comp_m4, 20)
print(f"\n{'='*40}\nM4 — ARR={ARR}% | HDA={HDA}% | DER={DER}% | PT={PT}s\n{'='*40}")
with open("resultados_compliance/resultados_comp_m4.json","w",encoding="utf-8") as f:
    json.dump({"modelo":"M4_mixto_compliance","metricas":{"ARR":ARR,"HDA":HDA,"DER":DER,"PT":PT},
               "resultados_por_caso":resultados_comp_m4}, f, ensure_ascii=False, indent=2)
print("✅ Guardado resultados_comp_m4.json")

Ejecutando M4 — Multi-Agent mixto (20 casos)...

  C01... renovacion_aprobada (auto)
  C02... renovacion_aprobada (auto)
  C03... renovacion_aprobada (auto)
  C04... renovacion_aprobada (auto)
  C05... renovacion_aprobada (auto)
  C06... renovacion_aprobada (auto)
  C07... renovacion_aprobada (auto)
  C08... renovacion_aprobada (auto)
  C09... renovacion_condicional (auto)
  C10... renovacion_condicional (auto)
  C11... renovacion_condicional (HiL)
  C12... renovacion_condicional (auto)
  C13... renovacion_aprobada (auto)
  C14... renovacion_condicional (auto)
  C15... renovacion_bloqueada (HiL)
  C16... escalacion_hil (HiL)
  C17... renovacion_condicional (HiL)
  C18... renovacion_bloqueada (HiL)
  C19... renovacion_condicional (HiL)
  C20... renovacion_bloqueada (HiL)

M4 — ARR=65.0% | HDA=100.0% | DER=35.0% | PT=3.56s
✅ Guardado resultados_comp_m4.json


M4 detecta los 6 casos HiL perfectamente (HDA=100%) gracias al modelo 70b en el agente decisor. ARR=65% ligeramente inferior a M2 — el sistema multi-agente es más conservador en compliance. DER mejora 5pp respecto a M2.

## Resumen — Experimento 04

In [10]:
import json

def calcular_metricas(resultados, total):
    resueltos = sum(1 for r in resultados if not r["escalo_hil"] and r["decision"]!="error")
    hil_req = [r for r in resultados if r["ground_truth"].get("requiere_hil")]
    hil_det = [r for r in hil_req if r["escalo_hil"]]
    errores = [r for r in resultados if r["decision"]!=r["ground_truth"].get("resultado","") and not r["escalo_hil"] and r["decision"]!="error"]
    ARR = round(resueltos/total*100,1)
    HDA = round(len(hil_det)/len(hil_req)*100,1) if hil_req else 0
    DER = round(len(errores)/total*100,1)
    PT  = round(sum(r["tiempo"] for r in resultados)/total,2)
    return ARR, HDA, DER, PT

print("="*50)
print("RESUMEN EXPERIMENTO 04 — COMPLIANCE CHECK")
print("="*50)
print(f"{'Modelo':<20} {'ARR':>6} {'HDA':>6} {'DER':>6} {'PT':>8}")
print("-"*50)
for nombre, archivo in [("M1 RPA", "resultados_compliance/resultados_comp_m1.json"),
                         ("M2 Single 8b", "resultados_compliance/resultados_comp_m2.json"),
                         ("M4 Multi mixto", "resultados_compliance/resultados_comp_m4.json")]:
    with open(archivo, encoding="utf-8") as f:
        d = json.load(f)
    a,h,de,p = calcular_metricas(d["resultados_por_caso"], 20)
    print(f"{nombre:<20} {a:>5}% {h:>5}% {de:>5}% {p:>7}s")
print("="*50)

RESUMEN EXPERIMENTO 04 — COMPLIANCE CHECK
Modelo                  ARR    HDA    DER       PT
--------------------------------------------------
M1 RPA               100.0%   0.0%  70.0%     0.0s
M2 Single 8b          70.0%  83.3%  40.0%     0.6s
M4 Multi mixto        65.0% 100.0%  35.0%    3.56s


Las señales de riesgo en compliance son mixtas — explícitas (sanciones OFAC, brecha GDPR, fraude) e implícitas (conflicto de interés accionista, accidente laboral). Esto produce HDA=83.3% en M2 y HDA=100% en M4, consistente con el gradiente de señales documentado en los experimentos anteriores.